# 02. OpenVLA LoRA Fine-tuning Skeleton

**필요 GPU**: A100 40GB 권장. T4 16GB는 batch_size=1 + gradient accumulation으로 가능하지만 매우 느림.

이 노트북은 **연구용 LoRA 파인튜닝 골격**입니다. 본격 학습 전에 다음을 검증합니다:
1. LoRA 어댑터가 정상 attach되는지
2. 100 step만 돌렸을 때 loss가 감소하는지
3. 어댑터를 저장/로드할 때 동작이 보존되는지

**중요**: 우리 연구의 진짜 학습은 04 노트북(GNN)이 메인이고, 여기는 baseline 비교용 OpenVLA 파인튜닝입니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
PROJECT_ROOT = '/content/drive/MyDrive/openvla-pose'
sys.path.insert(0, PROJECT_ROOT)
from openvla_helpers import load_openvla, extract_hidden_state
!nvidia-smi | head -10

In [ ]:
!pip install -q peft==0.11.1 bitsandbytes==0.43.1 trl==0.8.6 datasets==2.19.1
!pip install -q transformers==4.40.1 accelerate==0.27.2

## 1. Load model with 4-bit quantization (T4도 가능하게)

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForVision2Seq, BitsAndBytesConfig

HF_CACHE = f'{PROJECT_ROOT}/hf_cache'
os.environ['HF_HOME'] = HF_CACHE

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained('openvla/openvla-7b', trust_remote_code=True, cache_dir=HF_CACHE)
vla = AutoModelForVision2Seq.from_pretrained(
    'openvla/openvla-7b',
    quantization_config=bnb_config,
    attn_implementation='eager',
    trust_remote_code=True,
    cache_dir=HF_CACHE,
    device_map='auto',
)
print('Loaded in 4-bit')

## 2. Attach LoRA adapter

OpenVLA 권장 설정: rank=32, alpha=16, all linear layers.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

vla = prepare_model_for_kbit_training(vla)

lora_config = LoraConfig(
    r=32,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules='all-linear',
    init_lora_weights='gaussian',
)
vla = get_peft_model(vla, lora_config)
vla.print_trainable_parameters()

## 3. Tiny dummy dataset (smoke test)

실제 학습에선 LeRobot이나 Open X-Embodiment를 씁니다. 여기선 100 step 돌릴 더미만.

In [ ]:
from PIL import Image
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class DummyVLADataset(Dataset):
    def __init__(self, n=64):
        self.n = n
        self.instructions = ['pick up the cup', 'open the drawer', 'push the block', 'stack the cubes']
    def __len__(self):
        return self.n
    def __getitem__(self, i):
        rng = np.random.RandomState(i)
        arr = rng.randint(0, 255, (224, 224, 3), dtype=np.uint8)
        img = Image.fromarray(arr)
        inst = self.instructions[i % len(self.instructions)]
        action_str = ' '.join([str(rng.randint(0, 256)) for _ in range(7)])
        prompt = f'In: What action should the robot take to {inst}?\nOut: {action_str}'
        return {'image': img, 'prompt': prompt}

def collate(batch):
    imgs = [b['image'] for b in batch]
    prompts = [b['prompt'] for b in batch]
    inputs = processor(prompts, imgs, return_tensors='pt', padding=True)
    inputs['labels'] = inputs['input_ids'].clone()
    return inputs

loader = DataLoader(DummyVLADataset(64), batch_size=1, shuffle=True, collate_fn=collate)

## 4. Tiny training loop (loss 감소 확인)

In [ ]:
from torch.optim import AdamW

optim = AdamW([p for p in vla.parameters() if p.requires_grad], lr=5e-4)
vla.train()

losses = []
for step, batch in enumerate(loader):
    batch = {k: v.to(vla.device) for k, v in batch.items()}
    with torch.cuda.amp.autocast(dtype=torch.bfloat16):
        out = vla(**batch)
    loss = out.loss
    loss.backward()
    optim.step(); optim.zero_grad()
    losses.append(loss.item())
    if step % 5 == 0:
        print(f'step {step:3d}  loss {loss.item():.4f}')
    if step >= 30:
        break

import matplotlib.pyplot as plt
plt.plot(losses); plt.xlabel('step'); plt.ylabel('loss'); plt.title('LoRA smoke test')
plt.savefig(f'{PROJECT_ROOT}/data/lora_smoke_loss.png'); plt.show()

## 5. Save adapter to Drive

In [ ]:
ADAPTER_DIR = f'{PROJECT_ROOT}/checkpoints/openvla_lora_smoke'
vla.save_pretrained(ADAPTER_DIR)
print('Saved to', ADAPTER_DIR)
!ls -la $ADAPTER_DIR